# Pretrained image-to-3D backends (Colab Free)
Run all. TripoSR is the default real backend; Stable Fast 3D and Hunyuan3D are optional. The smoke test is skipped on a CPU runtime and never substitutes a fake mesh.

In [ ]:
# Single setup cell: project dependencies, GPU/VRAM check, and TripoSR source.
import importlib.util, subprocess, sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / 'image-to-3d-shapenet', *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'configs' / 'inference.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Clone the repository, then run this notebook from the repository or its parent.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
import torch
CUDA = torch.cuda.is_available()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA else 0.0
print({'cuda': CUDA, 'gpu': torch.cuda.get_device_name(0) if CUDA else None, 'vram_gb': round(VRAM_GB, 2)})
BACKEND_ROOT = PROJECT_ROOT / '.cache' / 'backends'
TRIPOSR_ROOT = BACKEND_ROOT / 'TripoSR'
if CUDA and importlib.util.find_spec('tsr') is None:
    BACKEND_ROOT.mkdir(parents=True, exist_ok=True)
    if not TRIPOSR_ROOT.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSR.git', str(TRIPOSR_ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf==2.3.0', 'einops==0.7.0', 'imageio[ffmpeg]', 'xatlas==0.0.9', 'moderngl==5.10.0', 'git+https://github.com/tatsy/torchmcubes.git'], check=True)
if TRIPOSR_ROOT.is_dir() and str(TRIPOSR_ROOT) not in sys.path:
    sys.path.insert(0, str(TRIPOSR_ROOT))
print('Setup complete. Advanced backends are intentionally not installed by default.')

In [ ]:
# A deterministic RGBA input; output still comes from real pretrained weights.
from PIL import Image, ImageDraw
input_path = PROJECT_ROOT / 'outputs' / 'pretrained_smoke_input.png'
input_path.parent.mkdir(parents=True, exist_ok=True)
image = Image.new('RGBA', (256, 256), (0, 0, 0, 0))
draw = ImageDraw.Draw(image)
draw.rounded_rectangle((55, 45, 200, 220), radius=20, fill=(145, 110, 75, 255))
draw.rectangle((75, 155, 92, 240), fill=(110, 80, 55, 255))
draw.rectangle((165, 155, 182, 240), fill=(110, 80, 55, 255))
image.save(input_path)
display(image)

In [ ]:
# Real-model smoke test. CPU runtimes skip cleanly; no synthetic output mesh is emitted.
if not CUDA:
    print('SKIP: select a Colab GPU runtime for the real pretrained smoke test.')
else:
    from image_to_3d_shapenet import load_config
    from image_to_3d_shapenet.inference import run_inference
    config = load_config(PROJECT_ROOT / 'configs' / 'inference.yaml').inference
    result = run_inference(input_path, PROJECT_ROOT / 'outputs' / 'pretrained_smoke', config)
    assert result['vertex_count'] > 0 and result['face_count'] > 0
    print(result)